# Briscola RL Demo: Training, Evaluation, Diagnostics

This notebook is a compact project demo. It does not reproduce the full experimental protocol. It shows one complete path:

1. load selected trained checkpoints;
2. optionally rerun training with explicit parameters;
3. run the fixed evaluation suite;
4. plot training and evaluation results;
5. optionally generate a diagnostics game and open the diagnostics UI.


## 1. Setup

Run this notebook from anywhere inside the repository. The next cell finds the project root and checks the expected scripts.


In [ ]:
from __future__ import annotations

import json
import math
import shlex
import subprocess
import sys
from html import escape
from pathlib import Path

from IPython.display import HTML, display


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "train.py").exists() and (candidate / "scripts" / "evaluate.py").exists():
            return candidate
    raise RuntimeError("Project root not found")


PROJECT_ROOT = find_project_root()
PYTHON_BIN = sys.executable

REQUIRED_PATHS = [
    PROJECT_ROOT / "scripts" / "train.py",
    PROJECT_ROOT / "scripts" / "evaluate.py",
    PROJECT_ROOT / "scripts" / "make_evaluation_graphs.py",
    PROJECT_ROOT / "scripts" / "record_diagnostics.py",
    PROJECT_ROOT / "diagnostics_ui" / "visual_diagnostics.html",
]

missing = [path for path in REQUIRED_PATHS if not path.exists()]
if missing:
    raise FileNotFoundError("Missing expected project files: " + ", ".join(str(path) for path in missing))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python used by notebook: {PYTHON_BIN}")


## 2. Demo switches

Default mode uses existing checkpoints and reruns only evaluation. Training and diagnostics UI are opt-in.


In [ ]:
LOAD_TRAINED_FROM_CHECKPOINT = True
RERUN_TRAINING = False
RUN_EVALUATION = True
RUN_SNAPSHOT_ARCHIVE_EVALUATION = False
RUN_DIAGNOSTICS_UI = False

EVALUATION_GAMES = 500
SEED_AMBIENTE_START = 100_000
SEED_POLICY_START = 200_000

SNAPSHOT_EVALUATION_GAMES = 100
SNAPSHOT_SEED_AMBIENTE_START = 900_000
SNAPSHOT_SEED_POLICY_START = 950_000
SNAPSHOT_ENTRY_STRIDE = 1
SNAPSHOT_MAX_ENTRIES = None

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Demo switches")
print(f"LOAD_TRAINED_FROM_CHECKPOINT={LOAD_TRAINED_FROM_CHECKPOINT}")
print(f"RERUN_TRAINING={RERUN_TRAINING}")
print(f"RUN_EVALUATION={RUN_EVALUATION}")
print(f"RUN_SNAPSHOT_ARCHIVE_EVALUATION={RUN_SNAPSHOT_ARCHIVE_EVALUATION}")
print(f"RUN_DIAGNOSTICS_UI={RUN_DIAGNOSTICS_UI}")


## 3. Selected checkpoints and configurations

The default checkpoints point to the selected linear and neural runs. Training from the notebook is optional and uses the same run parameters.


In [ ]:
LINEAR_PRETRAINED_CHECKPOINT = PROJECT_ROOT / "models" / "feature_set_new_aligned" / "learning_rate_0_9" / "combined_terminal" / "reward_alpha_0_2_lambda_margin_1_0" / "batch_size_300_updates_500" / "baseline_time_dependent_snapshot_interval_5_pool_size_20" / "matchup_sampling_per_rotation_block" / "bootstrap_updates_30" / "seed_5000" / "checkpoint.json"
LINEAR_PRETRAINED_LOG = LINEAR_PRETRAINED_CHECKPOINT.with_name("train_log.jsonl")

NEURAL_PRETRAINED_CHECKPOINT = PROJECT_ROOT / "models" / "feature_set_common_atomic" / "learning_rate_0_003" / "policy_neural_hidden_size_64_simple_reinforce_baseline" / "combined_terminal" / "reward_alpha_0_5_lambda_margin_0_5" / "batch_size_300_updates_500" / "baseline_time_dependent_snapshot_interval_5_pool_size_20" / "matchup_sampling_per_rotation_block" / "bootstrap_updates_30" / "seed_5000" / "checkpoint.json"
NEURAL_PRETRAINED_LOG = NEURAL_PRETRAINED_CHECKPOINT.with_name("train_log.jsonl")

LINEAR_DEMO_DIR = OUTPUT_DIR / "linear_new_aligned_demo"
NEURAL_DEMO_DIR = OUTPUT_DIR / "neural_demo"

LINEAR_DEMO_CHECKPOINT = LINEAR_DEMO_DIR / "checkpoint.json"
LINEAR_DEMO_LOG = LINEAR_DEMO_DIR / "train_log.jsonl"
NEURAL_DEMO_CHECKPOINT = NEURAL_DEMO_DIR / "checkpoint.json"
NEURAL_DEMO_LOG = NEURAL_DEMO_DIR / "train_log.jsonl"

LINEAR_TRAIN_CONFIG = {
    "policy_type": "linear",
    "feature_set": "new_aligned",
    "seed": 5000,
    "batch_size": 300,
    "updates": 500,
    "snapshot_interval": 5,
    "max_pool_size": 20,
    "warm_start_updates": 30,
    "keep_initial_pool": True,
    "matchup_sampling": "per_rotation_block",
    "learning_rate": 0.9,
    "baseline": "time_dependent",
    "reward_mode": "combined_terminal",
    "reward_alpha": 0.2,
    "reward_lambda_margin": 1.0,
}

NEURAL_TRAIN_CONFIG = {
    "policy_type": "neural",
    "feature_set": "common_atomic",
    "seed": 5000,
    "batch_size": 300,
    "updates": 500,
    "snapshot_interval": 5,
    "max_pool_size": 20,
    "warm_start_updates": 30,
    "keep_initial_pool": True,
    "matchup_sampling": "per_rotation_block",
    "learning_rate": 0.003,
    "hidden_size": 64,
    "baseline": "time_dependent",
    "neural_learned_baseline": False,
    "entropy_coef": 0.0,
    "reward_mode": "combined_terminal",
    "reward_alpha": 0.5,
    "reward_lambda_margin": 0.5,
}


def selected_paths(label: str) -> tuple[Path | None, Path | None]:
    use_demo_output = RERUN_TRAINING or not LOAD_TRAINED_FROM_CHECKPOINT
    if label == "linear":
        if use_demo_output:
            return LINEAR_DEMO_CHECKPOINT, LINEAR_DEMO_LOG
        return LINEAR_PRETRAINED_CHECKPOINT, LINEAR_PRETRAINED_LOG
    if label == "neural":
        if use_demo_output:
            return NEURAL_DEMO_CHECKPOINT, NEURAL_DEMO_LOG
        return NEURAL_PRETRAINED_CHECKPOINT, NEURAL_PRETRAINED_LOG
    raise ValueError(label)


for label in ("linear", "neural"):
    checkpoint, train_log = selected_paths(label)
    print(f"{label}: checkpoint={checkpoint}")
    print(f"{label}: train_log={train_log}")


## 4. Optional training

This cell calls the real training script. It is skipped unless `RERUN_TRAINING=True`.


In [ ]:
def train_command(config: dict, checkpoint_path: Path, log_path: Path) -> list[str]:
    command = [
        PYTHON_BIN,
        "-B",
        "scripts/train.py",
        "--policy-type",
        config["policy_type"],
        "--feature-set",
        config["feature_set"],
        "--seed",
        str(config["seed"]),
        "--batch-size",
        str(config["batch_size"]),
        "--updates",
        str(config["updates"]),
        "--snapshot-interval",
        str(config["snapshot_interval"]),
        "--max-pool-size",
        str(config["max_pool_size"]),
        "--warm-start-updates",
        str(config["warm_start_updates"]),
        "--matchup-sampling",
        config["matchup_sampling"],
        "--learning-rate",
        str(config["learning_rate"]),
        "--baseline",
        config["baseline"],
        "--reward-mode",
        config["reward_mode"],
        "--reward-alpha",
        str(config["reward_alpha"]),
        "--reward-lambda-margin",
        str(config["reward_lambda_margin"]),
        "--output",
        str(checkpoint_path),
        "--log",
        str(log_path),
    ]
    if config.get("keep_initial_pool"):
        command.append("--keep-initial-pool")
    if config["policy_type"] == "neural":
        command.extend(["--hidden-size", str(config["hidden_size"])])
        command.extend(["--entropy-coef", str(config["entropy_coef"])])
        if config.get("neural_learned_baseline", True):
            command.append("--neural-learned-baseline")
        else:
            command.append("--no-neural-learned-baseline")
    return command


def run_command(command: list[str]) -> None:
    print("$", " ".join(shlex.quote(part) for part in command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)


if RERUN_TRAINING:
    for label, config, checkpoint_path, log_path in [
        ("linear", LINEAR_TRAIN_CONFIG, LINEAR_DEMO_CHECKPOINT, LINEAR_DEMO_LOG),
        ("neural", NEURAL_TRAIN_CONFIG, NEURAL_DEMO_CHECKPOINT, NEURAL_DEMO_LOG),
    ]:
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Training {label} policy")
        run_command(train_command(config, checkpoint_path, log_path))
else:
    print("Training skipped. Set RERUN_TRAINING=True to run it from this notebook.")


## 5. Evaluation

Evaluation uses the same seed starts used in the saved project reports: `100000` for the environment and `200000` for policy tie-breaking/sampling.


In [ ]:
def evaluate_command(label: str, checkpoint_path: Path, output_path: Path) -> list[str]:
    return [
        PYTHON_BIN,
        "-B",
        "scripts/evaluate.py",
        "--checkpoint",
        str(checkpoint_path),
        "--games",
        str(EVALUATION_GAMES),
        "--seed-ambiente-start",
        str(SEED_AMBIENTE_START),
        "--seed-policy-start",
        str(SEED_POLICY_START),
        "--output",
        str(output_path),
    ]


reports: dict[str, Path] = {}
for label in ("linear", "neural"):
    checkpoint_path, _ = selected_paths(label)
    if checkpoint_path is None:
        print(f"Skipping {label}: checkpoint path is not configured yet.")
        continue
    if not checkpoint_path.exists():
        print(f"Skipping {label}: checkpoint does not exist: {checkpoint_path}")
        continue
    report_path = OUTPUT_DIR / f"evaluation_{label}_games_{EVALUATION_GAMES}.json"
    reports[label] = report_path
    if RUN_EVALUATION:
        run_command(evaluate_command(label, checkpoint_path, report_path))
    else:
        print(f"Evaluation skipped for {label}. Existing report expected at {report_path}")

print("Reports:")
for label, path in reports.items():
    print(label, path)


## 6. Load reports and show tables


In [ ]:
def load_evaluation_report(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))


def flatten_report(label: str, report: dict) -> list[dict]:
    rows = []
    for scenario in report["scenarios"]:
        metrics = scenario["metrics"]
        ci_low, ci_high = metrics["confidence_interval_95"]
        rows.append(
            {
                "model": label,
                "scenario": scenario["name"],
                "games": metrics["games"],
                "win_rate": metrics["win_rate"],
                "draw_rate": metrics["draw_rate"],
                "loss_rate": metrics["loss_rate"],
                "mean_margin": metrics["mean_point_difference"],
                "stderr": metrics["standard_error"],
                "ci95_low": ci_low,
                "ci95_high": ci_high,
            }
        )
    return rows


def render_table(rows: list[dict]) -> None:
    if not rows:
        print("No rows to display.")
        return
    columns = ["model", "scenario", "games", "win_rate", "draw_rate", "loss_rate", "mean_margin", "stderr", "ci95_low", "ci95_high"]
    head = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body = []
    for row in rows:
        cells = []
        for column in columns:
            value = row[column]
            if isinstance(value, float):
                value = f"{value:.3f}"
            cells.append(f"<td>{escape(str(value))}</td>")
        body.append("<tr>" + "".join(cells) + "</tr>")
    html = """
    <style>
      table.demo-table { border-collapse: collapse; font-family: system-ui, sans-serif; font-size: 13px; }
      .demo-table th, .demo-table td { border: 1px solid #ddd; padding: 4px 7px; text-align: right; }
      .demo-table th:nth-child(1), .demo-table td:nth-child(1),
      .demo-table th:nth-child(2), .demo-table td:nth-child(2) { text-align: left; }
      .demo-table th { background: #f5f5f5; }
    </style>
    """
    html += f"<table class='demo-table'><thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table>"
    display(HTML(html))


evaluation_rows = []
for label, report_path in reports.items():
    if report_path.exists():
        evaluation_rows.extend(flatten_report(label, load_evaluation_report(report_path)))

render_table(evaluation_rows)


## 7. Two-learner team evaluation

This optional check evaluates a full trained team: two learner checkpoints play together as players `0` and `2` against fixed opponent policies as players `1` and `3`. It answers a different question from the standard suite: not how one learner behaves with a fixed partner, but how a homogeneous learned pair performs as a team.


In [ ]:
from evaluation import play_match
from scripts.evaluate import load_checkpoint, learner_from_checkpoint
from policy import (
    AdvancedHeuristicPolicy,
    GreedyPolicy,
    HeuristicPolicy,
    PerfectHeuristicPolicy,
    RandomPolicy,
)


RUN_TWO_LEARNER_TEAM_EVALUATION = False
TWO_LEARNER_GAMES = 1000
TWO_LEARNER_SEED_AMBIENTE_START = 700_000
TWO_LEARNER_SEED_POLICY_START = 800_000


def opponent_pair(name: str):
    """Return the fixed opponent pair used against our two-learner team."""

    normalized = name.lower().strip()
    if normalized == "random":
        return RandomPolicy(), RandomPolicy()
    if normalized == "greedy":
        return GreedyPolicy(), GreedyPolicy()
    if normalized == "heuristic":
        return HeuristicPolicy(), HeuristicPolicy()
    if normalized == "advanced":
        return AdvancedHeuristicPolicy(), AdvancedHeuristicPolicy()
    if normalized == "perfect":
        return PerfectHeuristicPolicy(), PerfectHeuristicPolicy()
    if normalized == "advanced_perfect":
        return AdvancedHeuristicPolicy(), PerfectHeuristicPolicy()
    raise ValueError(f"Unknown opponent pair: {name}")


def load_learner_policy(checkpoint_path: Path):
    """Load either a linear or neural learner checkpoint using project code."""

    return learner_from_checkpoint(load_checkpoint(checkpoint_path))


def mean(values: list[float]) -> float:
    return sum(values) / len(values)


def sample_standard_error(values: list[float]) -> float:
    """Compute the standard error of match margins."""

    if len(values) < 2:
        return 0.0
    avg = mean(values)
    variance = sum((value - avg) ** 2 for value in values) / (len(values) - 1)
    return math.sqrt(variance / len(values))


def evaluate_two_learner_team(
    *,
    team_name: str,
    checkpoint_a: Path,
    checkpoint_b: Path,
    opponents: str,
    games: int = TWO_LEARNER_GAMES,
    seed_ambiente_start: int = TWO_LEARNER_SEED_AMBIENTE_START,
    seed_policy_start: int = TWO_LEARNER_SEED_POLICY_START,
    greedy: bool = True,
) -> dict:
    """Evaluate two trained learners as team `pari` against a fixed opponent pair."""

    learner_a = load_learner_policy(checkpoint_a)
    learner_b = load_learner_policy(checkpoint_b)
    opponent_a, opponent_b = opponent_pair(opponents)
    policies_by_player = {
        0: learner_a,
        2: learner_b,
        1: opponent_a,
        3: opponent_b,
    }

    margins: list[float] = []
    wins = draws = losses = 0
    for index in range(games):
        result = play_match(
            policies_by_player=policies_by_player,
            seed_ambiente=seed_ambiente_start + index,
            seed_policy=seed_policy_start + index,
            primo_giocatore_id=index % 4,
            greedy=greedy,
        )
        margin = float(result.margine_squadra_pari)
        margins.append(margin)
        if margin > 0:
            wins += 1
        elif margin < 0:
            losses += 1
        else:
            draws += 1

    stderr = sample_standard_error(margins)
    avg_margin = mean(margins)
    return {
        "team": team_name,
        "opponents": opponents,
        "games": games,
        "greedy": greedy,
        "win_rate": wins / games,
        "draw_rate": draws / games,
        "loss_rate": losses / games,
        "mean_margin": avg_margin,
        "standard_error": stderr,
        "ci95_low": avg_margin - 1.96 * stderr,
        "ci95_high": avg_margin + 1.96 * stderr,
        "checkpoint_a": str(checkpoint_a),
        "checkpoint_b": str(checkpoint_b),
        "seed_ambiente_start": seed_ambiente_start,
        "seed_policy_start": seed_policy_start,
    }


def display_two_learner_results(rows: list[dict]) -> None:
    """Show compact results for the two-learner team check."""

    if not rows:
        print("No two-learner team results to display.")
        return
    html_rows = "".join(
        "<tr>"
        f"<td>{escape(row['team'])}</td>"
        f"<td>{escape(row['opponents'])}</td>"
        f"<td>{row['games']}</td>"
        f"<td>{row['win_rate']:.3f}</td>"
        f"<td>{row['draw_rate']:.3f}</td>"
        f"<td>{row['loss_rate']:.3f}</td>"
        f"<td>{row['mean_margin']:.2f}</td>"
        f"<td>{row['standard_error']:.2f}</td>"
        f"<td>[{row['ci95_low']:.2f}, {row['ci95_high']:.2f}]</td>"
        "</tr>"
        for row in rows
    )
    display(HTML(f"""
    <table>
      <thead>
        <tr>
          <th>team</th><th>opponents</th><th>games</th><th>win</th><th>draw</th>
          <th>loss</th><th>mean margin</th><th>stderr</th><th>CI95</th>
        </tr>
      </thead>
      <tbody>{html_rows}</tbody>
    </table>
    """))


TWO_LEARNER_SCENARIOS = [
    ("linear_pair", LINEAR_PRETRAINED_CHECKPOINT, LINEAR_PRETRAINED_CHECKPOINT),
]
if NEURAL_PRETRAINED_CHECKPOINT is not None:
    TWO_LEARNER_SCENARIOS.append(
        ("neural_pair", NEURAL_PRETRAINED_CHECKPOINT, NEURAL_PRETRAINED_CHECKPOINT)
    )

TWO_LEARNER_OPPONENTS = ["advanced", "perfect", "advanced_perfect"]

two_learner_results: list[dict] = []
if RUN_TWO_LEARNER_TEAM_EVALUATION:
    for team_name, checkpoint_a, checkpoint_b in TWO_LEARNER_SCENARIOS:
        if not checkpoint_a.exists() or not checkpoint_b.exists():
            print(f"Skipping {team_name}: missing checkpoint")
            continue
        for opponents in TWO_LEARNER_OPPONENTS:
            two_learner_results.append(
                evaluate_two_learner_team(
                    team_name=team_name,
                    checkpoint_a=checkpoint_a,
                    checkpoint_b=checkpoint_b,
                    opponents=opponents,
                )
            )

    output_path = OUTPUT_DIR / "two_learner_team_evaluation.json"
    output_path.write_text(json.dumps(two_learner_results, indent=2), encoding="utf-8")
    print(f"saved_two_learner_team_evaluation={output_path}")

display_two_learner_results(two_learner_results)


## 8. Presentation-ready evaluation graphs

For each available candidate checkpoint, this section creates three standalone SVG graphs from the evaluation report: win rate, mean point difference with CI95, and win/draw/loss composition.


In [ ]:
def make_graphs_command(report_path: Path, out_dir: Path) -> list[str]:
    return [
        PYTHON_BIN,
        "-B",
        "scripts/make_evaluation_graphs.py",
        "--report",
        str(report_path),
        "--out-dir",
        str(out_dir),
    ]


def display_svg_file(path: Path) -> None:
    if path.exists():
        display(HTML(path.read_text(encoding="utf-8")))
    else:
        print(f"Missing SVG: {path}")


graph_dirs: dict[str, Path] = {}
for label, report_path in reports.items():
    if not report_path.exists():
        print(f"Skipping graphs for {label}: missing report {report_path}")
        continue
    out_dir = OUTPUT_DIR / f"graphs_{label}"
    graph_dirs[label] = out_dir
    run_command(make_graphs_command(report_path, out_dir))
    display(HTML(f"<h3>{escape(label)} evaluation graphs</h3>"))
    for name in (
        "hist_win_rate.svg",
        "hist_mean_point_difference.svg",
        "hist_outcome_composition.svg",
    ):
        display_svg_file(out_dir / name)


## 9. Comparison plots

These plots compare the selected final linear and neural candidates when both reports are available.


In [ ]:
def grouped_bar_svg(rows: list[dict], metric: str, title: str, value_label: str) -> str:
    if not rows:
        return "<p>No data available.</p>"
    scenarios = list(dict.fromkeys(row["scenario"] for row in rows))
    models = list(dict.fromkeys(row["model"] for row in rows))
    values = {(row["model"], row["scenario"]): float(row[metric]) for row in rows}
    max_value = max(abs(value) for value in values.values()) or 1.0
    min_value = min(values.values())
    baseline = 0 if min_value < 0 else None

    width = 1000
    row_height = 34
    top = 45
    left = 320
    plot_width = 560
    height = top + len(scenarios) * row_height + 40
    colors = ["#2f6fbb", "#c4512e", "#5f8f3a", "#7d5fb2"]
    parts = [f"<svg width='{width}' height='{height}' viewBox='0 0 {width} {height}' xmlns='http://www.w3.org/2000/svg'>"]
    parts.append(f"<text x='20' y='25' font-size='18' font-family='system-ui'>{escape(title)}</text>")
    if baseline is None:
        zero_x = left
        scale = plot_width / max_value
    else:
        zero_x = left + plot_width / 2
        scale = (plot_width / 2) / max_value
        parts.append(f"<line x1='{zero_x}' y1='{top-12}' x2='{zero_x}' y2='{height-25}' stroke='#888' stroke-width='1'/>")
    for i, scenario in enumerate(scenarios):
        y = top + i * row_height
        parts.append(f"<text x='20' y='{y + 18}' font-size='12' font-family='system-ui'>{escape(scenario)}</text>")
        for j, model in enumerate(models):
            value = values.get((model, scenario), 0.0)
            bar_height = max(7, row_height / (len(models) + 1))
            bar_y = y + 4 + j * bar_height
            if baseline is None:
                bar_x = zero_x
                bar_w = value * scale
            else:
                bar_w = abs(value) * scale
                bar_x = zero_x if value >= 0 else zero_x - bar_w
            color = colors[j % len(colors)]
            parts.append(f"<rect x='{bar_x:.1f}' y='{bar_y:.1f}' width='{bar_w:.1f}' height='{bar_height - 2:.1f}' fill='{color}'/>")
            label_x = bar_x + bar_w + 5 if value >= 0 else bar_x - 48
            parts.append(f"<text x='{label_x:.1f}' y='{bar_y + bar_height - 3:.1f}' font-size='11' font-family='system-ui'>{value:.3f}</text>")
    legend_x = left
    legend_y = height - 12
    for j, model in enumerate(models):
        x = legend_x + j * 120
        parts.append(f"<rect x='{x}' y='{legend_y - 10}' width='10' height='10' fill='{colors[j % len(colors)]}'/>")
        parts.append(f"<text x='{x + 15}' y='{legend_y}' font-size='12' font-family='system-ui'>{escape(model)}</text>")
    parts.append(f"<text x='{left}' y='{height - 30}' font-size='11' font-family='system-ui' fill='#555'>{escape(value_label)}</text>")
    parts.append("</svg>")
    return "".join(parts)


display(HTML(grouped_bar_svg(evaluation_rows, "win_rate", "Evaluation win rate", "higher is better")))
display(HTML(grouped_bar_svg(evaluation_rows, "mean_margin", "Evaluation mean point margin", "higher is better")))


## 10. Training log plots


In [ ]:
def load_jsonl(path: Path | None) -> list[dict]:
    if path is None or not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def line_svg(points: list[dict], metric: str, title: str) -> str:
    if not points:
        return f"<p>No training log available for {escape(title)}.</p>"
    xs = [float(row["update_index"]) for row in points]
    ys = [float(row[metric]) for row in points if metric in row]
    if not ys:
        return f"<p>Metric {escape(metric)} not available.</p>"
    values = [(float(row["update_index"]), float(row[metric])) for row in points if metric in row]
    min_x, max_x = min(x for x, _ in values), max(x for x, _ in values)
    min_y, max_y = min(y for _, y in values), max(y for _, y in values)
    if math.isclose(min_y, max_y):
        min_y -= 1.0
        max_y += 1.0
    width, height = 760, 250
    left, right, top, bottom = 55, 20, 35, 35
    plot_w = width - left - right
    plot_h = height - top - bottom
    def sx(x: float) -> float:
        return left + (x - min_x) / (max_x - min_x or 1.0) * plot_w
    def sy(y: float) -> float:
        return top + (max_y - y) / (max_y - min_y) * plot_h
    path_data = " ".join(("M" if i == 0 else "L") + f"{sx(x):.1f},{sy(y):.1f}" for i, (x, y) in enumerate(values))
    return f"""
    <svg width='{width}' height='{height}' viewBox='0 0 {width} {height}' xmlns='http://www.w3.org/2000/svg'>
      <text x='20' y='22' font-size='17' font-family='system-ui'>{escape(title)}</text>
      <rect x='{left}' y='{top}' width='{plot_w}' height='{plot_h}' fill='white' stroke='#ddd'/>
      <path d='{path_data}' fill='none' stroke='#2f6fbb' stroke-width='2'/>
      <text x='20' y='{top + 5}' font-size='11' font-family='system-ui'>{max_y:.3f}</text>
      <text x='20' y='{top + plot_h}' font-size='11' font-family='system-ui'>{min_y:.3f}</text>
      <text x='{left}' y='{height - 10}' font-size='11' font-family='system-ui'>update {int(min_x)}</text>
      <text x='{left + plot_w - 70}' y='{height - 10}' font-size='11' font-family='system-ui'>update {int(max_x)}</text>
    </svg>
    """


for label in ("linear", "neural"):
    _, train_log = selected_paths(label)
    rows = load_jsonl(train_log)
    if not rows:
        print(f"No training log available for {label}.")
        continue
    display(HTML(f"<h3>{escape(label)}</h3>"))
    for metric in ("mean_return", "mean_score_margin", "gradient_norm"):
        display(HTML(line_svg(rows, metric, f"{label}: {metric}")))


## 11. Snapshot archive evaluation

This section uses checkpoints saved by `scripts/train.py --snapshot-output-dir`. It evaluates archived snapshots over time on a small fixed subset of scenarios: `random_eval`, `greedy_eval`, and `advanced_heuristic_eval`.


In [ ]:
from evaluation import EvaluationScenario, EvaluationSuite, evaluate_suite, make_evaluation_cases
from policy import AdvancedHeuristicPolicy, GreedyPolicy, RandomPolicy
from scripts.evaluate import learner_from_checkpoint, load_checkpoint


SNAPSHOT_ARCHIVE_MANIFESTS = {
    "linear": PROJECT_ROOT / "models" / "snapshot_archive_runs" / "linear_new_aligned_lr0_9_alpha0_2_lambda1_0_warm_start30_seed5000" / "snapshots" / "snapshot_manifest.jsonl",
    "neural": PROJECT_ROOT / "models" / "snapshot_archive_runs" / "neural_common_atomic_lr0_003_h64_simple_baseline_alpha0_5_lambda0_5_warm_start30_seed5000" / "snapshots" / "snapshot_manifest.jsonl",
}
SNAPSHOT_SCENARIO_NAMES = ("random_eval", "greedy_eval", "advanced_heuristic_eval")


def snapshot_evaluation_suite() -> EvaluationSuite:
    return EvaluationSuite(
        scenarios=(
            EvaluationScenario(
                name="random_eval",
                compagno_policy=RandomPolicy(),
                avversario_successivo_policy=RandomPolicy(),
                avversario_precedente_policy=RandomPolicy(),
            ),
            EvaluationScenario(
                name="greedy_eval",
                compagno_policy=GreedyPolicy(),
                avversario_successivo_policy=GreedyPolicy(),
                avversario_precedente_policy=GreedyPolicy(),
            ),
            EvaluationScenario(
                name="advanced_heuristic_eval",
                compagno_policy=AdvancedHeuristicPolicy(),
                avversario_successivo_policy=AdvancedHeuristicPolicy(),
                avversario_precedente_policy=AdvancedHeuristicPolicy(),
            ),
        )
    )


def load_snapshot_manifest(manifest_path: Path) -> list[dict]:
    if not manifest_path.exists():
        return []
    entries = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return sorted(entries, key=lambda entry: int(entry["update_index"]))


def resolve_snapshot_checkpoint(manifest_path: Path, raw_path: str) -> Path:
    checkpoint_path = Path(raw_path)
    if checkpoint_path.is_absolute():
        return checkpoint_path
    project_path = PROJECT_ROOT / checkpoint_path
    if project_path.exists():
        return project_path
    return manifest_path.parent / checkpoint_path.name


def selected_snapshot_entries(entries: list[dict]) -> list[dict]:
    if SNAPSHOT_ENTRY_STRIDE <= 0:
        raise ValueError("SNAPSHOT_ENTRY_STRIDE deve essere positivo")
    selected = entries[::SNAPSHOT_ENTRY_STRIDE]
    if entries and selected[-1] is not entries[-1]:
        selected.append(entries[-1])
    if SNAPSHOT_MAX_ENTRIES is not None:
        selected = selected[:SNAPSHOT_MAX_ENTRIES]
    return selected


def snapshot_cache_path(label: str) -> Path:
    return OUTPUT_DIR / f"snapshot_archive_{label}_games_{SNAPSHOT_EVALUATION_GAMES}.jsonl"


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("".join(json.dumps(row) + "\n" for row in rows), encoding="utf-8")


def snapshot_metric_rows(*, label: str, entry: dict, results: dict) -> list[dict]:
    rows = []
    for scenario_name in SNAPSHOT_SCENARIO_NAMES:
        result = results[scenario_name]
        metrics = result.metrics
        ci_low, ci_high = metrics.confidence_interval_95
        rows.append(
            {
                "model": label,
                "update_index": int(entry["update_index"]),
                "kind": entry["kind"],
                "scenario": scenario_name,
                "games": metrics.games,
                "win_rate": metrics.win_rate,
                "draw_rate": metrics.draw_rate,
                "loss_rate": metrics.loss_rate,
                "mean_margin": metrics.mean_point_difference,
                "stderr": metrics.standard_error,
                "ci95_low": ci_low,
                "ci95_high": ci_high,
                "checkpoint_path": entry["checkpoint_path"],
            }
        )
    return rows


def evaluate_snapshot_archive(label: str, manifest_path: Path) -> list[dict]:
    entries = selected_snapshot_entries(load_snapshot_manifest(manifest_path))
    if not entries:
        print(f"No snapshot manifest found for {label}: {manifest_path}")
        return []
    suite = snapshot_evaluation_suite()
    cases = make_evaluation_cases(
        games=SNAPSHOT_EVALUATION_GAMES,
        seed_ambiente_start=SNAPSHOT_SEED_AMBIENTE_START,
        seed_policy_start=SNAPSHOT_SEED_POLICY_START,
    )
    rows: list[dict] = []
    for index, entry in enumerate(entries, start=1):
        checkpoint_path = resolve_snapshot_checkpoint(manifest_path, entry["checkpoint_path"])
        print(f"{label}: snapshot {index}/{len(entries)} update={entry['update_index']} checkpoint={checkpoint_path.name}")
        learner = learner_from_checkpoint(load_checkpoint(checkpoint_path))
        results = evaluate_suite(
            learner_policy=learner,
            suite=suite,
            cases=cases,
            learner_giocatore_id=0,
            greedy=True,
        )
        rows.extend(snapshot_metric_rows(label=label, entry=entry, results=results))
    return rows


def render_snapshot_summary(rows: list[dict]) -> None:
    if not rows:
        print("No snapshot evaluation rows available.")
        return
    latest_update = max(row["update_index"] for row in rows)
    latest_rows = [row for row in rows if row["update_index"] == latest_update]
    display(HTML(f"<h4>Latest archived snapshot: update {latest_update}</h4>"))
    render_table(latest_rows)


def snapshot_curve_svg(rows: list[dict], *, model: str, metric: str, title: str) -> str:
    model_rows = [row for row in rows if row["model"] == model]
    if not model_rows:
        return f"<p>No snapshot rows available for {escape(model)}.</p>"
    scenarios = list(dict.fromkeys(row["scenario"] for row in model_rows))
    values_by_scenario = {
        scenario: sorted(
            (
                (float(row["update_index"]), float(row[metric]))
                for row in model_rows
                if row["scenario"] == scenario
            ),
            key=lambda item: item[0],
        )
        for scenario in scenarios
    }
    all_points = [point for points in values_by_scenario.values() for point in points]
    min_x, max_x = min(x for x, _ in all_points), max(x for x, _ in all_points)
    if metric == "win_rate":
        min_y, max_y = 0.0, 1.0
    else:
        raw_min = min(y for _, y in all_points)
        raw_max = max(y for _, y in all_points)
        padding = max(2.0, (raw_max - raw_min) * 0.12)
        min_y = min(0.0, raw_min - padding)
        max_y = max(0.0, raw_max + padding)
    width, height = 900, 330
    left, right, top, bottom = 70, 190, 42, 42
    plot_w = width - left - right
    plot_h = height - top - bottom
    colors = ["#2563eb", "#f97316", "#16a34a"]

    def sx(x: float) -> float:
        return left + (x - min_x) / (max_x - min_x or 1.0) * plot_w

    def sy(y: float) -> float:
        return top + (max_y - y) / (max_y - min_y or 1.0) * plot_h

    parts = [f"<svg width='{width}' height='{height}' viewBox='0 0 {width} {height}' xmlns='http://www.w3.org/2000/svg'>"]
    parts.append(f"<rect width='{width}' height='{height}' fill='white'/>")
    parts.append(f"<text x='20' y='25' font-size='18' font-family='system-ui'>{escape(title)}</text>")
    parts.append(f"<rect x='{left}' y='{top}' width='{plot_w}' height='{plot_h}' fill='white' stroke='#ddd'/>")
    zero_y = sy(0.0)
    if top <= zero_y <= top + plot_h:
        parts.append(f"<line x1='{left}' y1='{zero_y:.1f}' x2='{left + plot_w}' y2='{zero_y:.1f}' stroke='#999' stroke-width='1'/>")
    for scenario_index, scenario in enumerate(scenarios):
        points = values_by_scenario[scenario]
        color = colors[scenario_index % len(colors)]
        path_data = " ".join(("M" if i == 0 else "L") + f"{sx(x):.1f},{sy(y):.1f}" for i, (x, y) in enumerate(points))
        parts.append(f"<path d='{path_data}' fill='none' stroke='{color}' stroke-width='2.4'/>")
        for x, y in points:
            parts.append(f"<circle cx='{sx(x):.1f}' cy='{sy(y):.1f}' r='2.8' fill='{color}'/>")
        legend_y = top + 18 + scenario_index * 22
        parts.append(f"<rect x='{left + plot_w + 24}' y='{legend_y - 10}' width='11' height='11' fill='{color}'/>")
        parts.append(f"<text x='{left + plot_w + 42}' y='{legend_y}' font-size='12' font-family='system-ui'>{escape(scenario)}</text>")
    parts.append(f"<text x='20' y='{top + 5}' font-size='11' font-family='system-ui'>{max_y:.3f}</text>")
    parts.append(f"<text x='20' y='{top + plot_h}' font-size='11' font-family='system-ui'>{min_y:.3f}</text>")
    parts.append(f"<text x='{left}' y='{height - 12}' font-size='11' font-family='system-ui'>update {int(min_x)}</text>")
    parts.append(f"<text x='{left + plot_w - 82}' y='{height - 12}' font-size='11' font-family='system-ui'>update {int(max_x)}</text>")
    parts.append("</svg>")
    return "".join(parts)


snapshot_rows: list[dict] = []
for label, manifest_path in SNAPSHOT_ARCHIVE_MANIFESTS.items():
    cache_path = snapshot_cache_path(label)
    if RUN_SNAPSHOT_ARCHIVE_EVALUATION:
        rows = evaluate_snapshot_archive(label, manifest_path)
        write_jsonl(cache_path, rows)
        print(f"Saved snapshot evaluation cache for {label}: {cache_path}")
    else:
        rows = load_jsonl(cache_path)
        if rows:
            print(f"Loaded cached snapshot evaluation for {label}: {cache_path}")
        else:
            print(f"Snapshot evaluation skipped for {label}. Set RUN_SNAPSHOT_ARCHIVE_EVALUATION=True after creating {manifest_path}")
    snapshot_rows.extend(rows)

render_snapshot_summary(snapshot_rows)
for model in dict.fromkeys(row["model"] for row in snapshot_rows):
    display(HTML(f"<h3>{escape(model)} snapshot archive</h3>"))
    display(HTML(snapshot_curve_svg(snapshot_rows, model=model, metric="win_rate", title=f"{model}: win rate over archived snapshots")))
    display(HTML(snapshot_curve_svg(snapshot_rows, model=model, metric="mean_margin", title=f"{model}: mean margin over archived snapshots")))


## 12. Optional diagnostics UI

This step records one deterministic game and starts a local web server for the visual diagnostics page. It is disabled by default.


In [ ]:
DIAGNOSTICS_MODEL = "linear"
DIAGNOSTICS_PARTNER = "perfect"
DIAGNOSTICS_OPPONENTS = "perfect"
DIAGNOSTICS_SEED_AMBIENTE = 100_000
DIAGNOSTICS_SEED_POLICY = 200_000
DIAGNOSTICS_PRIMO_GIOCATORE_ID = 0
DIAGNOSTICS_PORT = 8765


def diagnostics_command(checkpoint_path: Path, output_path: Path) -> list[str]:
    return [
        PYTHON_BIN,
        "-B",
        "scripts/record_diagnostics.py",
        "--checkpoint",
        str(checkpoint_path),
        "--output",
        str(output_path),
        "--partner",
        DIAGNOSTICS_PARTNER,
        "--opponents",
        DIAGNOSTICS_OPPONENTS,
        "--seed-ambiente",
        str(DIAGNOSTICS_SEED_AMBIENTE),
        "--seed-policy",
        str(DIAGNOSTICS_SEED_POLICY),
        "--primo-giocatore-id",
        str(DIAGNOSTICS_PRIMO_GIOCATORE_ID),
    ]


if RUN_DIAGNOSTICS_UI:
    checkpoint_path, _ = selected_paths(DIAGNOSTICS_MODEL)
    if checkpoint_path is None or not checkpoint_path.exists():
        raise FileNotFoundError(f"No checkpoint available for diagnostics: {checkpoint_path}")
    diagnostics_path = OUTPUT_DIR / f"diagnostics_{DIAGNOSTICS_MODEL}.json"
    run_command(diagnostics_command(checkpoint_path, diagnostics_path))
    relative_json = diagnostics_path.relative_to(PROJECT_ROOT).as_posix()
    diagnostics_server = subprocess.Popen(
        [PYTHON_BIN, "-m", "http.server", str(DIAGNOSTICS_PORT), "--bind", "127.0.0.1"],
        cwd=PROJECT_ROOT,
    )
    url = f"http://127.0.0.1:{DIAGNOSTICS_PORT}/diagnostics_ui/visual_diagnostics.html?json=/{relative_json}"
    print("Diagnostics server started. Open:")
    print(url)
    print("Stop it later with: diagnostics_server.terminate()")
else:
    print("Diagnostics UI skipped. Set RUN_DIAGNOSTICS_UI=True to generate a decision log and open the local UI.")


## 13. Reading guide

This notebook is a runnable demo. The full model-selection protocol, multi-seed comparisons, and ablation choices are documented separately in the project notes and run logs.
